# ASL Hand Landmark Extraction Pipeline
Batch-processing pipeline using OpenCV, MediaPipe Hands, and Pandas. Includes a feature to sample a small percentage of the data for quick demonstrations.

In [1]:
# Install a specific version of MediaPipe
!pip install mediapipe==0.10.14

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 34.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 40.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
ydf 0.14.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.


In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Import dependencies
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os
from tqdm.notebook import tqdm

# Suppress potential TensorFlow logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ==========================================
# DEMONSTRATION CONFIGURATION 
# ==========================================
# Set this to 20 to run on just 20% of the dataset for your supervisor.
# Change this back to 100 later to process the full dataset.
SAMPLE_PERCENT = 100 


# Output naming logic so you don't overwrite your full run dataset
if SAMPLE_PERCENT < 100:
    OUTPUT_CSV_PATH = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent.csv'
else:
    OUTPUT_CSV_PATH = '../data/final_extracted_landmarks_full.csv'

# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
)

# Define paths per requirements
BASE_DRIVE_PATH = '../data/asl_processed'
TRAIN_DIR = os.path.join(BASE_DRIVE_PATH, 'train')
TEST_DIR = os.path.join(BASE_DRIVE_PATH, 'test')

# Dynamically fetch all 42 classes from the directory
import os
try:
    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
except FileNotFoundError:
    # Fallback to generic names if directory not found locally
    all_classes = [str(i) for i in range(10)] + [chr(i) for i in range(ord('A'), ord('Z')+1)] + ['class37', 'class38', 'class39', 'class40', 'class41', 'class42']
print(f"Total classes found: {len(all_classes)}")


Total classes found: 42


In [4]:
def extract_landmarks_from_directory(dataset_dir, target_classes, sample_percent=100):
    """
    Iterates over the dataset. If sample_percent < 100, it selects only a fraction of the images
    from each class to speed up demonstration runs.
    """
    dataset_landmarks = []
    
    if not os.path.exists(dataset_dir):
        print(f"Warning: directory not found -> {dataset_dir}")
        return dataset_landmarks
        
    for class_name in tqdm(target_classes, desc=f"Processing {os.path.basename(dataset_dir)}"):
        class_dir = os.path.join(dataset_dir, class_name)
        
        if not os.path.exists(class_dir):
            continue
            
        # Sort files to ensure consistency
        image_files = sorted(os.listdir(class_dir))
        
        # Apply percentage limit if active
        if sample_percent < 100:
            num_to_process = max(1, int(len(image_files) * (sample_percent / 100.0)))
            image_files = image_files[:num_to_process]
        
        run_desc = f"Class {class_name} ({len(image_files)} imgs)"
        for img_name in tqdm(image_files, desc=run_desc, leave=False):
            img_path = os.path.join(class_dir, img_name)
            
            try:
                img = cv2.imread(img_path)
                if img is None:
                    continue
                    
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                results = hands.process(img_rgb)
                
                if results.multi_hand_landmarks:
                    for hand_landmarks in results.multi_hand_landmarks:
                        landmarks_row = []
                        for lm in hand_landmarks.landmark:
                            landmarks_row.extend([lm.x, lm.y, lm.z])
                            
                        landmarks_row.append(class_name)
                        dataset_landmarks.append(landmarks_row)
                        
            except Exception as e:
                continue
                
    return dataset_landmarks

In [5]:
# ==========================================
# RUN CLASS 0 (Index 0)
# ==========================================
import gc

target_classes = [all_classes[0]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 0 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part0.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part0.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 0.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 0!")



--- RUNNING CLASS 0 ---
Processing class: 0


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class 0 (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class 0 (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part0.csv

DONE WITH CLASS 0!


In [6]:
# ==========================================
# RUN CLASS 1 (Index 1)
# ==========================================
import gc

target_classes = [all_classes[1]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 1 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part1.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part1.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 1.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 1!")



--- RUNNING CLASS 1 ---
Processing class: 1


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class 1 (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class 1 (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part1.csv

DONE WITH CLASS 1!


In [7]:
# ==========================================
# RUN CLASS 2 (Index 2)
# ==========================================
import gc

target_classes = [all_classes[2]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 2 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part2.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part2.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 2.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 2!")



--- RUNNING CLASS 2 ---
Processing class: 2


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class 2 (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class 2 (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part2.csv

DONE WITH CLASS 2!


In [8]:
# ==========================================
# RUN CLASS 3 (Index 3)
# ==========================================
import gc

target_classes = [all_classes[3]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 3 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part3.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part3.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 3.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 3!")



--- RUNNING CLASS 3 ---
Processing class: 3


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class 3 (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class 3 (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part3.csv

DONE WITH CLASS 3!


In [9]:
# ==========================================
# RUN CLASS 4 (Index 4)
# ==========================================
import gc

target_classes = [all_classes[4]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 4 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part4.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part4.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 4.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 4!")



--- RUNNING CLASS 4 ---
Processing class: 4


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class 4 (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class 4 (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part4.csv

DONE WITH CLASS 4!


In [10]:
# ==========================================
# RUN CLASS 5 (Index 5)
# ==========================================
import gc

target_classes = [all_classes[5]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 5 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part5.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part5.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 5.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 5!")



--- RUNNING CLASS 5 ---
Processing class: 5


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class 5 (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class 5 (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part5.csv

DONE WITH CLASS 5!


In [11]:
# ==========================================
# RUN CLASS 6 (Index 6)
# ==========================================
import gc

target_classes = [all_classes[6]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 6 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part6.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part6.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 6.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 6!")



--- RUNNING CLASS 6 ---
Processing class: 6


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class 6 (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class 6 (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part6.csv

DONE WITH CLASS 6!


In [12]:
# ==========================================
# RUN CLASS 7 (Index 7)
# ==========================================
import gc

target_classes = [all_classes[7]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 7 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part7.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part7.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 7.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 7!")



--- RUNNING CLASS 7 ---
Processing class: 7


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class 7 (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class 7 (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part7.csv

DONE WITH CLASS 7!


In [13]:
# ==========================================
# RUN CLASS 8 (Index 8)
# ==========================================
import gc

target_classes = [all_classes[8]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 8 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part8.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part8.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 8.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 8!")



--- RUNNING CLASS 8 ---
Processing class: 8


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class 8 (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class 8 (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part8.csv

DONE WITH CLASS 8!


In [14]:
# ==========================================
# RUN CLASS 9 (Index 9)
# ==========================================
import gc

target_classes = [all_classes[9]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 9 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part9.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part9.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 9.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 9!")



--- RUNNING CLASS 9 ---
Processing class: 9


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class 9 (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class 9 (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part9.csv

DONE WITH CLASS 9!


In [15]:
# ==========================================
# RUN CLASS 10 (Index 10)
# ==========================================
import gc

target_classes = [all_classes[10]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 10 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part10.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part10.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 10.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 10!")



--- RUNNING CLASS 10 ---
Processing class: A


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class A (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class A (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part10.csv

DONE WITH CLASS 10!


In [16]:
# ==========================================
# RUN CLASS 11 (Index 11)
# ==========================================
import gc

target_classes = [all_classes[11]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 11 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part11.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part11.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 11.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 11!")



--- RUNNING CLASS 11 ---
Processing class: B


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class B (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class B (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part11.csv

DONE WITH CLASS 11!


In [17]:
# ==========================================
# RUN CLASS 12 (Index 12)
# ==========================================
import gc

target_classes = [all_classes[12]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 12 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part12.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part12.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 12.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 12!")



--- RUNNING CLASS 12 ---
Processing class: Bye


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class Bye (320 imgs):   0%|          | 0/320 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class Bye (80 imgs):   0%|          | 0/80 [00:00<?, ?it/s]

No valid landmarks discovered for class 12.

DONE WITH CLASS 12!


In [18]:
# ==========================================
# RUN CLASS 13 (Index 13)
# ==========================================
import gc

target_classes = [all_classes[13]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 13 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part13.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part13.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 13.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 13!")



--- RUNNING CLASS 13 ---
Processing class: C


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class C (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class C (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part13.csv

DONE WITH CLASS 13!


In [19]:
# ==========================================
# RUN CLASS 14 (Index 14)
# ==========================================
import gc

target_classes = [all_classes[14]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 14 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part14.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part14.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 14.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 14!")



--- RUNNING CLASS 14 ---
Processing class: D


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class D (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class D (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part14.csv

DONE WITH CLASS 14!


In [20]:
# ==========================================
# RUN CLASS 15 (Index 15)
# ==========================================
import gc

target_classes = [all_classes[15]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 15 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part15.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part15.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 15.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 15!")



--- RUNNING CLASS 15 ---
Processing class: E


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class E (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class E (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part15.csv

DONE WITH CLASS 15!


In [5]:
# ==========================================
# RUN CLASS 16 (Index 16)
# ==========================================
import gc

target_classes = [all_classes[16]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 16 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part16.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part16.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 16.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 16!")



--- RUNNING CLASS 16 ---
Processing class: F


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class F (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class F (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part16.csv

DONE WITH CLASS 16!


In [5]:
# ==========================================
# RUN CLASS 17 (Index 17)
# ==========================================
import gc

target_classes = [all_classes[17]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 17 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part17.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part17.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 17.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 17!")



--- RUNNING CLASS 17 ---
Processing class: G


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class G (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class G (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part17.csv

DONE WITH CLASS 17!


In [6]:
# ==========================================
# RUN CLASS 18 (Index 18)
# ==========================================
import gc

target_classes = [all_classes[18]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 18 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part18.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part18.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 18.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 18!")



--- RUNNING CLASS 18 ---
Processing class: H


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class H (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class H (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part18.csv

DONE WITH CLASS 18!


In [7]:
# ==========================================
# RUN CLASS 19 (Index 19)
# ==========================================
import gc

target_classes = [all_classes[19]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 19 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part19.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part19.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 19.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 19!")



--- RUNNING CLASS 19 ---
Processing class: Hello


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class Hello (320 imgs):   0%|          | 0/320 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class Hello (80 imgs):   0%|          | 0/80 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part19.csv

DONE WITH CLASS 19!


In [8]:
# ==========================================
# RUN CLASS 20 (Index 20)
# ==========================================
import gc

target_classes = [all_classes[20]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 20 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part20.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part20.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 20.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 20!")



--- RUNNING CLASS 20 ---
Processing class: I


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class I (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class I (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part20.csv

DONE WITH CLASS 20!


In [9]:
# ==========================================
# RUN CLASS 21 (Index 21)
# ==========================================
import gc

target_classes = [all_classes[21]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 21 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part21.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part21.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 21.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 21!")



--- RUNNING CLASS 21 ---
Processing class: J


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class J (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class J (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part21.csv

DONE WITH CLASS 21!


In [10]:
# ==========================================
# RUN CLASS 22 (Index 22)
# ==========================================
import gc

target_classes = [all_classes[22]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 22 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part22.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part22.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 22.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 22!")



--- RUNNING CLASS 22 ---
Processing class: K


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class K (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class K (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part22.csv

DONE WITH CLASS 22!


In [11]:
# ==========================================
# RUN CLASS 23 (Index 23)
# ==========================================
import gc

target_classes = [all_classes[23]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 23 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part23.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part23.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 23.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 23!")



--- RUNNING CLASS 23 ---
Processing class: L


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class L (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class L (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part23.csv

DONE WITH CLASS 23!


In [12]:
# ==========================================
# RUN CLASS 24 (Index 24)
# ==========================================
import gc

target_classes = [all_classes[24]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 24 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part24.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part24.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 24.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 24!")



--- RUNNING CLASS 24 ---
Processing class: M


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class M (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class M (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part24.csv

DONE WITH CLASS 24!


In [13]:
# ==========================================
# RUN CLASS 25 (Index 25)
# ==========================================
import gc

target_classes = [all_classes[25]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 25 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part25.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part25.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 25.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 25!")



--- RUNNING CLASS 25 ---
Processing class: N


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class N (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class N (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part25.csv

DONE WITH CLASS 25!


In [14]:
# ==========================================
# RUN CLASS 26 (Index 26)
# ==========================================
import gc

target_classes = [all_classes[26]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 26 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part26.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part26.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 26.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 26!")



--- RUNNING CLASS 26 ---
Processing class: No


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class No (320 imgs):   0%|          | 0/320 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class No (80 imgs):   0%|          | 0/80 [00:00<?, ?it/s]

No valid landmarks discovered for class 26.

DONE WITH CLASS 26!


In [5]:
# ==========================================
# RUN CLASS 27 (Index 27)
# ==========================================
import gc

target_classes = [all_classes[27]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 27 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part27.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part27.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 27.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 27!")



--- RUNNING CLASS 27 ---
Processing class: O


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class O (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class O (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part27.csv

DONE WITH CLASS 27!


In [6]:
# ==========================================
# RUN CLASS 28 (Index 28)
# ==========================================
import gc

target_classes = [all_classes[28]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 28 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part28.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part28.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 28.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 28!")



--- RUNNING CLASS 28 ---
Processing class: P


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class P (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class P (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part28.csv

DONE WITH CLASS 28!


In [7]:
# ==========================================
# RUN CLASS 29 (Index 29)
# ==========================================
import gc

target_classes = [all_classes[29]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 29 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part29.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part29.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 29.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 29!")



--- RUNNING CLASS 29 ---
Processing class: Perfect


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class Perfect (320 imgs):   0%|          | 0/320 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class Perfect (80 imgs):   0%|          | 0/80 [00:00<?, ?it/s]

No valid landmarks discovered for class 29.

DONE WITH CLASS 29!


In [8]:
# ==========================================
# RUN CLASS 30 (Index 30)
# ==========================================
import gc

target_classes = [all_classes[30]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 30 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part30.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part30.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 30.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 30!")



--- RUNNING CLASS 30 ---
Processing class: Q


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class Q (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class Q (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part30.csv

DONE WITH CLASS 30!


In [5]:
# ==========================================
# RUN CLASS 31 (Index 31)
# ==========================================
import gc

target_classes = [all_classes[31]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 31 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part31.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part31.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 31.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 31!")



--- RUNNING CLASS 31 ---
Processing class: R


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class R (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class R (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part31.csv

DONE WITH CLASS 31!


In [5]:
# ==========================================
# RUN CLASS 32 (Index 32)
# ==========================================
import gc

target_classes = [all_classes[32]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 32 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part32.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part32.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 32.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 32!")



--- RUNNING CLASS 32 ---
Processing class: S


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class S (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class S (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part32.csv

DONE WITH CLASS 32!


In [6]:
# ==========================================
# RUN CLASS 33 (Index 33)
# ==========================================
import gc

target_classes = [all_classes[33]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 33 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part33.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part33.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 33.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 33!")



--- RUNNING CLASS 33 ---
Processing class: T


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class T (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class T (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part33.csv

DONE WITH CLASS 33!


In [7]:
# ==========================================
# RUN CLASS 34 (Index 34)
# ==========================================
import gc

target_classes = [all_classes[34]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 34 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part34.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part34.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 34.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 34!")



--- RUNNING CLASS 34 ---
Processing class: Thank You


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class Thank You (320 imgs):   0%|          | 0/320 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class Thank You (80 imgs):   0%|          | 0/80 [00:00<?, ?it/s]

No valid landmarks discovered for class 34.

DONE WITH CLASS 34!


In [8]:
# ==========================================
# RUN CLASS 35 (Index 35)
# ==========================================
import gc

target_classes = [all_classes[35]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 35 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part35.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part35.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 35.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 35!")



--- RUNNING CLASS 35 ---
Processing class: U


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class U (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class U (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part35.csv

DONE WITH CLASS 35!


In [9]:
# ==========================================
# RUN CLASS 36 (Index 36)
# ==========================================
import gc

target_classes = [all_classes[36]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 36 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part36.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part36.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 36.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 36!")



--- RUNNING CLASS 36 ---
Processing class: V


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class V (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class V (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part36.csv

DONE WITH CLASS 36!


In [10]:
# ==========================================
# RUN CLASS 37 (Index 37)
# ==========================================
import gc

target_classes = [all_classes[37]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 37 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part37.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part37.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 37.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 37!")



--- RUNNING CLASS 37 ---
Processing class: W


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class W (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class W (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part37.csv

DONE WITH CLASS 37!


In [5]:
# ==========================================
# RUN CLASS 38 (Index 38)
# ==========================================
import gc

target_classes = [all_classes[38]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 38 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part38.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part38.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 38.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 38!")



--- RUNNING CLASS 38 ---
Processing class: X


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class X (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class X (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part38.csv

DONE WITH CLASS 38!


In [6]:
# ==========================================
# RUN CLASS 39 (Index 39)
# ==========================================
import gc

target_classes = [all_classes[39]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 39 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part39.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part39.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 39.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 39!")



--- RUNNING CLASS 39 ---
Processing class: Y


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class Y (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class Y (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part39.csv

DONE WITH CLASS 39!


In [7]:
# ==========================================
# RUN CLASS 40 (Index 40)
# ==========================================
import gc

target_classes = [all_classes[40]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 40 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part40.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part40.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 40.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 40!")



--- RUNNING CLASS 40 ---
Processing class: Yes


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class Yes (320 imgs):   0%|          | 0/320 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class Yes (80 imgs):   0%|          | 0/80 [00:00<?, ?it/s]

No valid landmarks discovered for class 40.

DONE WITH CLASS 40!


In [8]:
# ==========================================
# RUN CLASS 41 (Index 41)
# ==========================================
import gc

target_classes = [all_classes[41]]
if not target_classes:
    print("No classes in this group.")
else:
    print(f"\n--- RUNNING CLASS 41 ---")
    print(f"Processing class: {target_classes[0]}")
    
    train_data = extract_landmarks_from_directory(TRAIN_DIR, target_classes, SAMPLE_PERCENT)
    test_data = extract_landmarks_from_directory(TEST_DIR, target_classes, SAMPLE_PERCENT)
    
    all_extracted_data = train_data + test_data
    
    if all_extracted_data:
        columns = []
        for j in range(21):
            columns.extend([f'x{j}', f'y{j}', f'z{j}'])
        columns.append('class_label')
        
        import pandas as pd
        df = pd.DataFrame(all_extracted_data, columns=columns)
        
        # Save part to CSV
        if SAMPLE_PERCENT < 100:
            part_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part41.csv'
        else:
            part_csv_path = f'../data/final_extracted_landmarks_full_part41.csv'
            
        df.to_csv(part_csv_path, index=False)
        print(f"Saved {{len(df)}} landmarks into {part_csv_path}")
    else:
        print(f"No valid landmarks discovered for class 41.")
        
    # Clear memory
    del train_data, test_data, all_extracted_data
    gc.collect()
    print(f"\nDONE WITH CLASS 41!")



--- RUNNING CLASS 41 ---
Processing class: Z


Processing train:   0%|          | 0/1 [00:00<?, ?it/s]

Class Z (800 imgs):   0%|          | 0/800 [00:00<?, ?it/s]

Processing test:   0%|          | 0/1 [00:00<?, ?it/s]

Class Z (200 imgs):   0%|          | 0/200 [00:00<?, ?it/s]

Saved {len(df)} landmarks into ../data/final_extracted_landmarks_full_part41.csv

DONE WITH CLASS 41!


In [10]:
# ==========================================
# MERGE ALL PARTS INTO A FULL DATASET 
# ==========================================
# Run this cell ONLY AFTER you have completed all runs (e.g. CURRENT_RUN_INDEX from 0 to 6)
# It will merge all the individual CSV parts into one single CSV file.

import pandas as pd
import glob
import os

if SAMPLE_PERCENT < 100:
    part_files_pattern = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_part*.csv'
    merged_csv_path = f'../data/extracted_landmarks_demo_{SAMPLE_PERCENT}percent_MERGED.csv'
else:
    part_files_pattern = '../data/final_extracted_landmarks_full_part*.csv'
    merged_csv_path = '../data/final_extracted_landmarks_full_MERGED.csv'

all_files = glob.glob(part_files_pattern)
print(f"Found {len(all_files)} part files.")

if len(all_files) > 0:
    df_list = []
    for file in all_files:
        print(f"Loading {file}...")
        df_list.append(pd.read_csv(file))
        
    merged_df = pd.concat(df_list, ignore_index=True)
    merged_df.to_csv(merged_csv_path, index=False)
    print(f"\nSUCCESS! Merged {len(all_files)} files with a total of {len(merged_df)} landmarks.")
    print(f"Saved to: {merged_csv_path}")
else:
    print("No part files found to merge.")


Found 36 part files.
Loading ../data/final_extracted_landmarks_full_part7.csv...
Loading ../data/final_extracted_landmarks_full_part15.csv...
Loading ../data/final_extracted_landmarks_full_part13.csv...
Loading ../data/final_extracted_landmarks_full_part11.csv...
Loading ../data/final_extracted_landmarks_full_part4.csv...
Loading ../data/final_extracted_landmarks_full_part8.csv...
Loading ../data/final_extracted_landmarks_full_part1.csv...
Loading ../data/final_extracted_landmarks_full_part2.csv...
Loading ../data/final_extracted_landmarks_full_part3.csv...
Loading ../data/final_extracted_landmarks_full_part10.csv...
Loading ../data/final_extracted_landmarks_full_part9.csv...
Loading ../data/final_extracted_landmarks_full_part0.csv...
Loading ../data/final_extracted_landmarks_full_part5.csv...
Loading ../data/final_extracted_landmarks_full_part6.csv...
Loading ../data/final_extracted_landmarks_full_part14.csv...
Loading ../data/final_extracted_landmarks_full_part32.csv...
Loading ../da